# 🌿 TerraMind Afrique — Fine-Tuning du Modèle de Diagnostic (`terramind_v1`)
**Responsable : Oscar (IA & Données)** · **PAS Challenge 2026 (SALTIS)**

Ce notebook est **100 % autonome** (aucun `git clone` requis, tous les scripts sont créés directement dans les cellules via `%%writefile`) :
1. **Cellule 1** : Vérifie le GPU T4, installe les paquets et crée les dossiers.
2. **Cellules 2 à 5** : Écrivent `data/label_mapping.csv`, `build_merged_dataset.py`, `split_dataset.py` et `train_classifier.py`.
3. **Cellule 6** : Télécharge TOM2024 (Burkina Faso) et CCMT Raw Data (Ghana) — ou les réutilise s'ils sont déjà présents dans `/content/data_local/raw`.
4. **Cellule 7** : Construit `terramind_v1` (35 448 images nettoyées par SHA-256) et le split stratifié `train / val / test` (80 / 10 / 10).
5. **Cellule 8** : Fine-tune **`linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification`** en FP16 (2 époques tête + 6 époques complètes) avec suivi de progression tous les 100 lots.
6. **Cellule 9** : Affiche le rapport d'évaluation (`ml/eval/metrics.json`) et télécharge automatiquement `metrics.json`, `classes.json` et `terramind_v1_model.pt` sur ton PC.


In [ ]:
# 1. Vérification du GPU, installation des dépendances et création des dossiers
!nvidia-smi
!pip install -q datasets transformers accelerate pillow torchvision kagglehub
!mkdir -p data/exclusions ml/training ml/eval modeles/diagnostic
print("Environnement prêt !")


In [ ]:
%%writefile data/label_mapping.csv
source,crop_src,label_src,crop,label,categorie,libelle_fr,note
tom2024,Maize,healthy-leaf,maize,healthy,sain,Maïs sain,fusionné avec CCMT Maize/healthy
tom2024,Maize,abiotic-disease-d,maize,abiotic_disorder,abiotique,Trouble abiotique,
tom2024,Maize,aphids-p,maize,aphids,ravageur,Pucerons,
tom2024,Maize,curvulariosis-d,maize,curvularia_leaf_spot,maladie,Curvulariose,
tom2024,Maize,helminthosporiosis-d,maize,helminthosporiosis,maladie,Helminthosporiose,à valider : peut-être la même maladie que CCMT Maize/leaf blight ; gardées séparées
tom2024,Maize,rust-d,maize,rust,maladie,Rouille,
tom2024,Maize,spodoptera-frugiperda-a,maize,fall_armyworm,ravageur,Chenille légionnaire,images de dégâts sur feuille (observation visuelle) ; fusionné
tom2024,Maize,spodoptera-frugiperda-p,maize,fall_armyworm,ravageur,Chenille légionnaire,images de la chenille (observation visuelle) ; fusionné
tom2024,Maize,stripe-d,maize,stripe,maladie,Stries (stripe),à valider : ressemble à CCMT Maize/streak virus mais virus différent ; gardées séparées
tom2024,Onion,healthy-leaf,onion,healthy,sain,Oignon sain,
tom2024,Onion,alternaria-d,onion,alternaria,maladie,Alternariose,
tom2024,Onion,bulb-blight-d,onion,bulb_blight,maladie,Pourriture du bulbe,
tom2024,Onion,caterpillar-p,onion,caterpillars,ravageur,Chenilles,
tom2024,Onion,fusarium-d,onion,fusarium,maladie,Fusariose,
tom2024,Onion,virosis-d,onion,virosis,maladie,Virose,
tom2024,Tomato,healthy-leaf,tomato,healthy,sain,Tomate saine (feuille),fusionné avec CCMT Tomato/healthy
tom2024,Tomato,healthy-fruit,tomato,healthy_fruit,sain,Fruit de tomate sain,photo de fruit et non de feuille
tom2024,Tomato,alternaria-d,tomato,alternaria,maladie,Alternariose,
tom2024,Tomato,alternaria-mite-d,tomato,alternaria_and_mites,mixte,Alternariose et acariens,
tom2024,Tomato,bacterial-floundering-d,tomato,bacterial_wilt,maladie,Flétrissement bactérien,
tom2024,Tomato,blossom-end-rot-d,tomato,blossom_end_rot,abiotique,Pourriture apicale,photo de fruit ; trouble physiologique et non maladie
tom2024,Tomato,excess-nitrogen-d,tomato,excess_nitrogen,abiotique,Excès d'azote,
tom2024,Tomato,fusarium-d,tomato,fusarium,maladie,Fusariose,
tom2024,Tomato,helicoverpa-armigera-p,tomato,helicoverpa_armigera,ravageur,Noctuelle de la tomate (Helicoverpa),
tom2024,Tomato,mite-d,tomato,mites,ravageur,Acariens,suffixe -d dans la source mais ravageur
tom2024,Tomato,sunburn-d,tomato,sunburn,abiotique,Coup de soleil,
tom2024,Tomato,tomato-late-blight-d,tomato,late_blight,maladie,Mildiou,
tom2024,Tomato,tuta-absoluta-p,tomato,tuta_absoluta,ravageur,Mineuse de la tomate (Tuta),
tom2024,Tomato,virosis-d,tomato,virosis,maladie,Virose,
ccmt,Cashew,healthy,cashew,healthy,sain,Cajou sain,
ccmt,Cashew,anthracnose,cashew,anthracnose,maladie,Anthracnose,
ccmt,Cashew,gumosis,cashew,gummosis,maladie,Gommose,
ccmt,Cashew,leaf miner,cashew,leaf_miner,ravageur,Mineuse des feuilles,
ccmt,Cashew,red rust,cashew,red_rust,maladie,Rouille rouge,
ccmt,Cassava,healthy,cassava,healthy,sain,Manioc sain,
ccmt,Cassava,bacterial blight,cassava,bacterial_blight,maladie,Bactériose,
ccmt,Cassava,brown spot,cassava,brown_spot,maladie,Taches brunes,
ccmt,Cassava,green mite,cassava,green_mite,ravageur,Acarien vert,
ccmt,Cassava,mosaic,cassava,mosaic,maladie,Mosaïque,
ccmt,Maize,healthy,maize,healthy,sain,Maïs sain,fusionné avec TOM2024 Maize/healthy-leaf
ccmt,Maize,fall armyworm,maize,fall_armyworm,ravageur,Chenille légionnaire,dégâts et chenilles mélangés ; fusionné avec TOM2024 spodoptera-frugiperda
ccmt,Maize,grasshoper,maize,grasshopper,ravageur,Criquets,coquille corrigée
ccmt,Maize,leaf beetle,maize,leaf_beetle,ravageur,Chrysomèle,
ccmt,Maize,leaf blight,maize,leaf_blight,maladie,Brûlure des feuilles,à valider : voir helminthosporiosis ; étiquettes peu fiables (D12)
ccmt,Maize,leaf spot,maize,leaf_spot,maladie,Taches foliaires,étiquettes peu fiables (D12)
ccmt,Maize,streak virus,maize,streak_virus,maladie,Virus de la striure,à valider : voir stripe
ccmt,Tomato,healthy,tomato,healthy,sain,Tomate saine (feuille),fusionné avec TOM2024 Tomato/healthy-leaf
ccmt,Tomato,leaf blight,tomato,leaf_blight,maladie,Brûlure des feuilles,à valider : alternariose ou mildiou ? gardée séparée
ccmt,Tomato,leaf curl,tomato,leaf_curl,maladie,Enroulement des feuilles,
ccmt,Tomato,septoria leaf spot,tomato,septoria_leaf_spot,maladie,Septoriose,
ccmt,Tomato,verticulium wilt,tomato,verticillium_wilt,maladie,Verticilliose,coquille corrigée


In [ ]:
%%writefile ml/training/build_merged_dataset.py
"""Construit la base fusionnée TerraMind (TOM2024 raw + CCMT Raw Data) depuis les copies locales.

Étapes (décisions D12 et D13 de docs/decisions.md) :
1. étiquettes uniformisées avec data/label_mapping.csv ;
2. exclusion des images illisibles, non JPEG ou dont le petit côté fait moins de MIN_SIDE px ;
3. doublons exacts (SHA-256) : une seule copie si même classe, toutes exclues si classes différentes ;
4. retrait des classes de moins de MIN_IMAGES images ;
5. copie des images (octets d'origine, sans réencodage) et écriture des manifestes.

À lancer depuis la racine du dépôt : python ml/training/build_merged_dataset.py
"""

from __future__ import annotations

import csv
import hashlib
import io
import sys
from collections import Counter, defaultdict
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

from datasets import Image as HFImage
from datasets import load_from_disk
from PIL import Image

TOM_DIR = Path("data_local/raw/tom2024_raw")
CCMT_DIR = Path("data_local/raw/ccmt_raw/CCMT Dataset")
MAPPING = Path("data/label_mapping.csv")
OUT = Path("data_local/processed/terramind_v1")
EXCLUSIONS_TRACKED = Path("data/exclusions/terramind_v1.csv")
MIN_SIDE = 64
MIN_IMAGES = 50


@dataclass
class Record:
    source: str
    ref: str
    crop_src: str
    label_src: str
    load: Callable[[], bytes]
    sha256: str = ""
    width: int = 0
    height: int = 0
    crop: str = ""
    label: str = ""
    reason: str = ""
    file: str = ""


def tom2024_records() -> list[Record]:
    ds = load_from_disk(str(TOM_DIR))
    names = ds.features["label"].names
    raw = ds.cast_column("image", HFImage(decode=False))
    crops, labels = ds["crop_type"], ds["label"]
    return [
        Record(
            "tom2024",
            f"row{i:05d}",
            crops[i],
            names[labels[i]],
            lambda i=i: raw[i]["image"]["bytes"],
        )
        for i in range(ds.num_rows)
    ]


def ccmt_records() -> list[Record]:
    records = []
    for path in sorted(CCMT_DIR.rglob("*.jpg")):
        crop, label = path.relative_to(CCMT_DIR).parts[:2]
        ref = "/".join(path.relative_to(CCMT_DIR).parts)
        records.append(Record("ccmt", ref, crop, label, path.read_bytes))
    return records


def inspect(record: Record) -> None:
    data = record.load()
    record.sha256 = hashlib.sha256(data).hexdigest()
    try:
        with Image.open(io.BytesIO(data)) as image:
            image.verify()
        with Image.open(io.BytesIO(data)) as image:
            image.load()
            record.width, record.height = image.size
            image_format = image.format
    except Exception:
        record.reason = "illisible"
        return
    if image_format != "JPEG":
        record.reason = "format_non_jpeg"
    elif min(record.width, record.height) < MIN_SIDE:
        record.reason = f"petit_cote_moins_de_{MIN_SIDE}px"


def mark_duplicates(records: list[Record]) -> None:
    groups = defaultdict(list)
    for record in records:
        if not record.reason:
            groups[record.sha256].append(record)
    for group in groups.values():
        if len(group) < 2:
            continue
        if len({(r.crop, r.label) for r in group}) > 1:
            for record in group:
                record.reason = "doublon_etiquettes_differentes"
        else:
            for record in group[1:]:
                record.reason = "doublon_meme_classe"


def main() -> int:
    if OUT.exists():
        print(f"{OUT} existe déjà : le supprimer ou le renommer avant de reconstruire.")
        return 1
    with MAPPING.open(encoding="utf-8") as handle:
        mapping = {(m["source"], m["crop_src"], m["label_src"]): m for m in csv.DictReader(handle)}

    records = tom2024_records() + ccmt_records()
    records.sort(key=lambda r: (r.source, r.ref))
    for record in records:
        target = mapping.get((record.source, record.crop_src, record.label_src))
        if target is None:
            print(
                f"Étiquette absente de {MAPPING} : {record.source} {record.crop_src}/{record.label_src}"
            )
            return 1
        record.crop, record.label = target["crop"], target["label"]
        inspect(record)
    mark_duplicates(records)

    counts = Counter((r.crop, r.label) for r in records if not r.reason)
    for record in records:
        if not record.reason and counts[(record.crop, record.label)] < MIN_IMAGES:
            record.reason = f"classe_moins_de_{MIN_IMAGES}_images"
    kept = [r for r in records if not r.reason]

    for record in kept:
        record.file = (
            f"images/{record.crop}/{record.label}/{record.source}_{record.sha256[:16]}.jpg"
        )
        target = OUT / record.file
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(record.load())

    labels = {(m["crop"], m["label"]): m for m in mapping.values()}
    with (OUT / "manifest.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(
            [
                "file",
                "crop",
                "label",
                "categorie",
                "source",
                "source_ref",
                "crop_src",
                "label_src",
                "sha256",
                "width",
                "height",
            ]
        )
        for r in kept:
            writer.writerow(
                [
                    r.file,
                    r.crop,
                    r.label,
                    labels[(r.crop, r.label)]["categorie"],
                    r.source,
                    r.ref,
                    r.crop_src,
                    r.label_src,
                    r.sha256,
                    r.width,
                    r.height,
                ]
            )

    per_source = Counter((r.crop, r.label, r.source) for r in kept)
    with (OUT / "classes.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(["crop", "label", "categorie", "libelle_fr", "images", "tom2024", "ccmt"])
        for crop, label in sorted(Counter((r.crop, r.label) for r in kept)):
            info = labels[(crop, label)]
            tom, ccmt = per_source[(crop, label, "tom2024")], per_source[(crop, label, "ccmt")]
            writer.writerow(
                [crop, label, info["categorie"], info["libelle_fr"], tom + ccmt, tom, ccmt]
            )

    excluded = [r for r in records if r.reason]
    for path in (OUT / "exclusions.csv", EXCLUSIONS_TRACKED):
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.writer(handle, lineterminator="\n")
            writer.writerow(["source", "source_ref", "crop_src", "label_src", "raison"])
            writer.writerows([r.source, r.ref, r.crop_src, r.label_src, r.reason] for r in excluded)

    print(f"images lues : {len(records)} | conservées : {len(kept)} | exclues : {len(excluded)}")
    for reason, n in Counter(r.reason for r in excluded).most_common():
        print(f"  exclues, {reason} : {n}")
    print(f"classes : {len(counts) - sum(1 for n in counts.values() if n < MIN_IMAGES)}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ml/training/split_dataset.py
"""Découpage stratifié et reproductible de terramind_v1 en train (80 %), val (10 %) et test (10 %).

Stratification par (crop, label, source) après tri par empreinte SHA-256 :
- garantit que chaque classe (même les plus petites comme tomato/late_blight à 72 images)
  est présente dans train, val et test dans les mêmes proportions ;
- garantit que les deux sources (TOM2024 et CCMT) sont équilibrées dans chaque split ;
- marque les 2 classes de fruits de la tomate (`healthy_fruit`, `blossom_end_rot`) pour
  permettre un entraînement strictement « feuilles » (`--leaf-only`) ou complet.

Entrée : data_local/processed/terramind_v1/manifest.csv
Sorties :
- data_local/processed/terramind_v1/splits.csv (hors Git, 35 448 lignes avec colonne `split`)
- data/splits_terramind_v1.csv (suivi dans Git : effectifs train/val/test par classe)

À lancer depuis la racine :
    .venv\\Scripts\\python.exe ml/training/split_dataset.py
"""

from __future__ import annotations

import csv
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

MANIFEST = Path("data_local/processed/terramind_v1/manifest.csv")
CLASSES_CSV = Path("data_local/processed/terramind_v1/classes.csv")
SPLITS_OUT = Path("data_local/processed/terramind_v1/splits.csv")
SUMMARY_OUT = Path("data/splits_terramind_v1.csv")

SEED = 42
VAL_RATIO = 0.10
TEST_RATIO = 0.10
CLASSES_FRUITS = {("tomato", "healthy_fruit"), ("tomato", "blossom_end_rot")}


def assigner_splits(lignes: list[dict], seed: int = SEED) -> list[dict]:
    """Assigne 'train', 'val' ou 'test' par strate (crop, label, source)."""
    strates: dict[tuple[str, str, str], list[dict]] = defaultdict(list)
    for row in sorted(lignes, key=lambda r: r["sha256"]):
        clef = (row["crop"], row["label"], row["source"])
        strates[clef].append(row)

    resultat: list[dict] = []
    for clef in sorted(strates):
        groupe = list(strates[clef])
        rng = random.Random(f"{seed}:{clef[0]}:{clef[1]}:{clef[2]}")
        rng.shuffle(groupe)
        n = len(groupe)
        n_test = max(1, int(round(n * TEST_RATIO)))
        n_val = max(1, int(round(n * VAL_RATIO)))
        for idx, item in enumerate(groupe):
            copie = dict(item)
            if idx < n_test:
                copie["split"] = "test"
            elif idx < n_test + n_val:
                copie["split"] = "val"
            else:
                copie["split"] = "train"
            copie["est_feuille"] = (
                "non" if (copie["crop"], copie["label"]) in CLASSES_FRUITS else "oui"
            )
            resultat.append(copie)

    resultat.sort(key=lambda r: r["file"])
    return resultat


def main() -> int:
    if not MANIFEST.is_file():
        print(f"Manifeste introuvable : {MANIFEST}. Lancer d'abord build_merged_dataset.py.")
        return 1

    with MANIFEST.open(encoding="utf-8", newline="") as f:
        lignes = list(csv.DictReader(f))

    libelles: dict[tuple[str, str], str] = {}
    if CLASSES_CSV.is_file():
        with CLASSES_CSV.open(encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f):
                libelles[(r["crop"], r["label"])] = r["libelle_fr"]

    avec_splits = assigner_splits(lignes)

    colonnes_splits = [
        "file",
        "split",
        "crop",
        "label",
        "categorie",
        "est_feuille",
        "source",
        "source_ref",
        "crop_src",
        "label_src",
        "sha256",
        "width",
        "height",
    ]
    with SPLITS_OUT.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes_splits, lineterminator="\n")
        writer.writeheader()
        writer.writerows(avec_splits)

    compteurs = Counter((r["crop"], r["label"], r["split"]) for r in avec_splits)
    sources_cnt = Counter((r["crop"], r["label"], r["source"]) for r in avec_splits)
    categories = {(r["crop"], r["label"]): r["categorie"] for r in avec_splits}

    SUMMARY_OUT.parent.mkdir(parents=True, exist_ok=True)
    with SUMMARY_OUT.open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(
            [
                "crop",
                "label",
                "categorie",
                "libelle_fr",
                "est_feuille",
                "train",
                "val",
                "test",
                "total",
                "tom2024",
                "ccmt",
            ]
        )
        for crop, label in sorted(categories):
            n_tr = compteurs[(crop, label, "train")]
            n_va = compteurs[(crop, label, "val")]
            n_te = compteurs[(crop, label, "test")]
            n_tom = sources_cnt[(crop, label, "tom2024")]
            n_ccmt = sources_cnt[(crop, label, "ccmt")]
            est_f = "non" if (crop, label) in CLASSES_FRUITS else "oui"
            w.writerow(
                [
                    crop,
                    label,
                    categories[(crop, label)],
                    libelles.get((crop, label), label),
                    est_f,
                    n_tr,
                    n_va,
                    n_te,
                    n_tr + n_va + n_te,
                    n_tom,
                    n_ccmt,
                ]
            )

    tot_split = Counter(r["split"] for r in avec_splits)
    print(
        f"Découpage terminé ({len(avec_splits)} images) : "
        f"train={tot_split['train']} | val={tot_split['val']} | test={tot_split['test']}"
    )
    print(f"Fichiers écrits : {SPLITS_OUT} et {SUMMARY_OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ml/training/train_classifier.py
"""Fine-tuning d'un modèle de vision pré-entraîné (PlantVillage HF ou MobileNetV3) sur terramind_v1.

Respecte la décision D5 (licence permissive Apache-2.0 / BSD, pas d'Ultralytics/YOLOv8 AGPL)
et la décision D13 (base fusionnée terramind_v1 : TOM2024 Burkina Faso + CCMT Ghana).

Deux backbones supportés (--backbone) :
1. `linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification` (défaut, via Hugging Face
   `transformers`) : MobileNetV2 déjà pré-entraîné sur 38 classes de maladies foliaires
   (PlantVillage), adapté ensuite aux vraies photos de champ ouest-africaines de terramind_v1.
2. `mobilenet_v3_large` (via `torchvision.models`, pré-entraîné ImageNet-1K).

Stratégie d'entraînement en 2 phases (avec précision mixte FP16 sur GPU T4) :
- Phase 1 (tête seule, backbone gelé, 2 époques, lr=1e-3)
- Phase 2 (fine-tuning complet avec CosineAnnealingLR, 6 époques, lr=1e-4)
- Pondération inversement proportionnelle à la racine de l'effectif (`1 / sqrt(n_c)`) pour
  protéger les petites classes (ex. mildiou de la tomate à 58 images d'entraînement).
- Redimensionnement systématique en 224x224 px pour neutraliser le biais de résolution
  entre CCMT (400x400) et TOM2024 (tailles variables).

Sorties :
- Poids et métadonnées des classes : `modeles/diagnostic/terramind_v1_model.pt` et `classes.json`
- Rapport complet d'évaluation et calibration du seuil « incertain » : `ml/eval/metrics.json`

Usage (sur Google Colab / Kaggle GPU ou machine avec PyTorch) :
    python ml/training/train_classifier.py --epochs-head 2 --epochs-finetune 6 --leaf-only
"""

from __future__ import annotations

import argparse
import csv
import datetime as dt
import json
import math
import time
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("data_local/processed/terramind_v1")
SPLITS_CSV = DATA_DIR / "splits.csv"
SUMMARY_CSV = Path("data/splits_terramind_v1.csv")
MODEL_OUT_DIR = Path("modeles/diagnostic")
METRICS_OUT = Path("ml/eval/metrics.json")

HF_DEFAULT_MODEL = "linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification"
IMAGE_SIZE = 224
PETITES_CLASSES_SEUIL_IMAGES = 100


def charger_splits(
    leaf_only: bool = True, cultures: set[str] | None = None
) -> tuple[list[dict], list[dict]]:
    """Lit splits.csv et construit la table ordonnée des classes (crop, label)."""
    if not SPLITS_CSV.is_file():
        raise FileNotFoundError(
            f"{SPLITS_CSV} introuvable. Lancer d'abord : python ml/training/split_dataset.py"
        )

    libelles: dict[tuple[str, str], str] = {}
    totaux_classe: dict[tuple[str, str], int] = {}
    if SUMMARY_CSV.is_file():
        with SUMMARY_CSV.open(encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f):
                clef = (r["crop"], r["label"])
                libelles[clef] = r["libelle_fr"]
                totaux_classe[clef] = int(r["total"])

    lignes: list[dict] = []
    with SPLITS_CSV.open(encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if leaf_only and row.get("est_feuille") == "non":
                continue
            if cultures and row["crop"] not in cultures:
                continue
            lignes.append(row)

    couples = sorted({(r["crop"], r["label"], r["categorie"]) for r in lignes})
    classes_info = []
    for idx, (crop, label, categorie) in enumerate(couples):
        total_img = totaux_classe.get((crop, label), 100)
        classes_info.append(
            {
                "index": idx,
                "cle": f"{crop}__{label}",
                "culture": crop,
                "etiquette": label,
                "categorie": categorie,
                "libelle": libelles.get((crop, label), label),
                "total_images": total_img,
                "petite_classe": total_img < PETITES_CLASSES_SEUIL_IMAGES,
            }
        )
    return lignes, classes_info


def calibrer_seuil_incertain(y_true: list[int], y_pred: list[int], confs: list[float]) -> dict:
    """Calcule la courbe Précision vs Couverture et choisit le seuil « incertain » optimal."""
    courbe = []
    seuil_retenu = 0.60
    for s_int in range(40, 95, 5):
        seuil = round(s_int / 100.0, 2)
        gardes = [
            (yt, yp) for yt, yp, c in zip(y_true, y_pred, confs, strict=False) if c >= seuil
        ]
        couverture = len(gardes) / max(1, len(y_true))
        exactitude = sum(1 for yt, yp in gardes if yt == yp) / len(gardes) if gardes else 0.0
        courbe.append(
            {
                "seuil": seuil,
                "exactitude_retenus": round(exactitude, 4),
                "taux_couverture": round(couverture, 4),
                "taux_incertain": round(1.0 - couverture, 4),
            }
        )
        if exactitude >= 0.85 and couverture >= 0.70 and seuil_retenu == 0.60:
            seuil_retenu = seuil

    return {"seuil_recommande": seuil_retenu, "courbe_seuil": courbe}


def entrainer(args: argparse.Namespace) -> int:
    try:
        import torch
        import torch.nn as nn
        from PIL import Image
        from torch.utils.data import DataLoader, Dataset
        from torchvision import transforms
    except ImportError as exc:
        print(
            f"PyTorch / torchvision requis pour l'entraînement ({exc}).\n"
            "Sur Google Colab ou Kaggle, ils sont déjà installés par défaut."
        )
        return 1

    cultures_filtre = (
        {c.strip().lower() for c in args.crops.split(",") if c.strip()} if args.crops else None
    )
    lignes, classes_info = charger_splits(leaf_only=args.leaf_only, cultures=cultures_filtre)
    cle_vers_idx = {(c["culture"], c["etiquette"]): c["index"] for c in classes_info}
    nb_classes = len(classes_info)

    print(f"Classes retenues : {nb_classes} | Images totales : {len(lignes)}", flush=True)

    tf_train = transforms.Compose(
        [
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    tf_eval = transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    class TerraMindDataset(Dataset):
        def __init__(self, items: list[dict], transform):
            self.items = items
            self.transform = transform

        def __len__(self) -> int:
            return len(self.items)

        def __getitem__(self, idx: int):
            r = self.items[idx]
            img_path = DATA_DIR / r["file"]
            with Image.open(img_path) as im:
                tenseur = self.transform(im.convert("RGB"))
            cible = cle_vers_idx[(r["crop"], r["label"])]
            return tenseur, cible, r["source"], r["crop"]

    train_items = [r for r in lignes if r["split"] == "train"]
    val_items = [r for r in lignes if r["split"] == "val"]
    test_items = [r for r in lignes if r["split"] == "test"]

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    print(f"Appareil d'entraînement : {device} | Backbone : {args.backbone}", flush=True)

    ds_train = TerraMindDataset(train_items, tf_train)
    ds_val = TerraMindDataset(val_items, tf_eval)
    ds_test = TerraMindDataset(test_items, tf_eval)

    dl_train = DataLoader(
        ds_train, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=use_cuda
    )
    dl_val = DataLoader(
        ds_val, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=use_cuda
    )
    dl_test = DataLoader(
        ds_test, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=use_cuda
    )

    util_hf = False
    if args.backbone != "mobilenet_v3_large":
        try:
            from transformers import AutoModelForImageClassification

            modele = AutoModelForImageClassification.from_pretrained(
                args.backbone,
                num_labels=nb_classes,
                ignore_mismatched_sizes=True,
            )
            util_hf = True
        except Exception as exc:
            print(f"Repli sur torchvision mobilenet_v3_large ({exc})", flush=True)

    if not util_hf:
        from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large

        modele = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_features = modele.classifier[-1].in_features
        modele.classifier[-1] = nn.Linear(in_features, nb_classes)

    modele.to(device)

    # Pondération par classe : 1 / sqrt(effectif_train)
    compte_train = Counter(cle_vers_idx[(r["crop"], r["label"])] for r in train_items)
    poids = torch.tensor(
        [1.0 / math.sqrt(max(1, compte_train[i])) for i in range(nb_classes)],
        dtype=torch.float32,
        device=device,
    )
    poids = poids / poids.mean()
    critere = nn.CrossEntropyLoss(weight=poids)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

    def _logits(sortie):
        return sortie.logits if hasattr(sortie, "logits") else sortie

    def evaluer(chargeur):
        modele.eval()
        y_true, y_pred, confs, sources, crops = [], [], [], [], []
        with torch.no_grad():
            for x, y, src, crp in chargeur:
                x = x.to(device, non_blocking=use_cuda)
                with torch.amp.autocast("cuda", enabled=use_cuda):
                    probs = torch.softmax(_logits(modele(x)), dim=-1)
                conf, pred = probs.max(dim=-1)
                y_true.extend(y.tolist())
                y_pred.extend(pred.cpu().tolist())
                confs.extend(conf.cpu().tolist())
                sources.extend(src)
                crops.extend(crp)
        return y_true, y_pred, confs, sources, crops

    def _boucle_epoque(opt, phase_nom: str, ep: int, ep_tot: int):
        modele.train()
        t0 = time.time()
        perte_cumul = 0.0
        nb_lots = len(dl_train)
        for i, (x, y, _, _) in enumerate(dl_train, start=1):
            x = x.to(device, non_blocking=use_cuda)
            y = y.to(device, non_blocking=use_cuda)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_cuda):
                perte = critere(_logits(modele(x)), y)
            scaler.scale(perte).backward()
            scaler.step(opt)
            scaler.update()
            perte_cumul += perte.item()
            if i % 100 == 0 or i == nb_lots:
                print(
                    f"  [{phase_nom} {ep}/{ep_tot}] lot {i}/{nb_lots} | "
                    f"loss={perte_cumul / i:.4f} ({time.time() - t0:.0f}s)",
                    flush=True,
                )

    # Phase 1 : entraînement de la tête seule
    if args.epochs_head > 0:
        for nom, param in modele.named_parameters():
            param.requires_grad = "classifier" in nom
        opt_head = torch.optim.AdamW(
            [p for p in modele.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4
        )
        for ep in range(1, args.epochs_head + 1):
            _boucle_epoque(opt_head, "Phase 1 - Tête", ep, args.epochs_head)
            yt, yp, _, _, _ = evaluer(dl_val)
            acc_val = sum(1 for a, b in zip(yt, yp, strict=False) if a == b) / max(1, len(yt))
            print(f"=> [Phase 1 - Tête] Époque {ep}/{args.epochs_head} | val_acc={acc_val:.4f}", flush=True)

    # Phase 2 : fine-tuning complet
    for param in modele.parameters():
        param.requires_grad = True
    opt_ft = torch.optim.AdamW(modele.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=max(1, args.epochs_finetune))

    meilleur_val = 0.0
    MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
    chemin_poids = MODEL_OUT_DIR / "terramind_v1_model.pt"

    for ep in range(1, args.epochs_finetune + 1):
        _boucle_epoque(opt_ft, "Phase 2 - FineTune", ep, args.epochs_finetune)
        sched.step()
        yt, yp, _, _, _ = evaluer(dl_val)
        acc_val = sum(1 for a, b in zip(yt, yp, strict=False) if a == b) / max(1, len(yt))
        print(f"=> [Phase 2 - FineTune] Époque {ep}/{args.epochs_finetune} | val_acc={acc_val:.4f}", flush=True)
        if acc_val >= meilleur_val:
            meilleur_val = acc_val
            torch.save(
                {
                    "backbone": args.backbone,
                    "util_hf": util_hf,
                    "nb_classes": nb_classes,
                    "classes": classes_info,
                    "state_dict": modele.state_dict(),
                },
                chemin_poids,
            )

    # Évaluation finale sur val (calibration seuil) et test (métriques publiées)
    yt_val, yp_val, conf_val, _, _ = evaluer(dl_val)
    calibration = calibrer_seuil_incertain(yt_val, yp_val, conf_val)
    seuil_opt = calibration["seuil_recommande"]

    yt_test, yp_test, conf_test, src_test, crp_test = evaluer(dl_test)
    acc_test_global = sum(1 for a, b in zip(yt_test, yp_test, strict=False) if a == b) / max(
        1, len(yt_test)
    )

    # Précision par source (TOM2024 vs CCMT) et par culture
    par_source = defaultdict(lambda: [0, 0])
    par_culture = defaultdict(lambda: [0, 0])
    par_classe = defaultdict(lambda: [0, 0])
    for yt, yp, src, crp in zip(yt_test, yp_test, src_test, crp_test, strict=False):
        ok = int(yt == yp)
        par_source[src][0] += ok
        par_source[src][1] += 1
        par_culture[crp][0] += ok
        par_culture[crp][1] += 1
        par_classe[yt][0] += ok
        par_classe[yt][1] += 1

    gardes_test = [
        (yt, yp) for yt, yp, c in zip(yt_test, yp_test, conf_test, strict=False) if c >= seuil_opt
    ]
    taux_incertain_test = 1.0 - (len(gardes_test) / max(1, len(yt_test)))
    acc_test_hors_incertain = (
        sum(1 for yt, yp in gardes_test if yt == yp) / len(gardes_test) if gardes_test else 0.0
    )

    rapport = {
        "date_evaluation": dt.date.today().isoformat(),
        "modele": args.backbone,
        "base": "terramind_v1 (TOM2024 + CCMT)",
        "nb_classes": nb_classes,
        "leaf_only": args.leaf_only,
        "effectifs": {
            "train": len(train_items),
            "val": len(val_items),
            "test": len(test_items),
        },
        "seuil_incertain": seuil_opt,
        "test_metrics": {
            "exactitude_globale": round(acc_test_global, 4),
            "exactitude_hors_incertain": round(acc_test_hors_incertain, 4),
            "taux_incertain": round(taux_incertain_test, 4),
            "par_source": {
                s: {"exactitude": round(ok / n, 4), "images": n}
                for s, (ok, n) in sorted(par_source.items())
            },
            "par_culture": {
                c: {"exactitude": round(ok / n, 4), "images": n}
                for c, (ok, n) in sorted(par_culture.items())
            },
            "par_classe": {
                classes_info[idx]["cle"]: {
                    "libelle": classes_info[idx]["libelle"],
                    "exactitude": round(ok / n, 4),
                    "images_test": n,
                }
                for idx, (ok, n) in sorted(par_classe.items())
            },
        },
        "calibration_val": calibration,
    }

    METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
    METRICS_OUT.write_text(json.dumps(rapport, indent=2, ensure_ascii=False), encoding="utf-8")
    (MODEL_OUT_DIR / "classes.json").write_text(
        json.dumps(classes_info, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Poids enregistrés dans {chemin_poids}", flush=True)
    print(f"Métriques enregistrées dans {METRICS_OUT}", flush=True)
    return 0


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Fine-tuning du modèle de diagnostic TerraMind v1.")
    p.add_argument("--backbone", default=HF_DEFAULT_MODEL)
    p.add_argument("--epochs-head", type=int, default=2)
    p.add_argument("--epochs-finetune", type=int, default=6)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--leaf-only", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument(
        "--crops",
        default="",
        help="Liste optionnelle de cultures séparées par des virgules (ex. tomato,onion,maize)",
    )
    return p


if __name__ == "__main__":
    raise SystemExit(entrainer(build_parser().parse_args()))


In [ ]:
# 6. Téléchargement rapide de TOM2024 et CCMT (sauté si déjà présents dans /content/data_local/raw)
from pathlib import Path
import shutil
from datasets import load_dataset
import kagglehub

tom_dir = Path("data_local/raw/tom2024_raw")
target_ccmt = Path("data_local/raw/ccmt_raw/CCMT Dataset")

if tom_dir.exists() and any(tom_dir.iterdir()):
    print("TOM2024 déjà présent dans data_local/raw/tom2024_raw, téléchargement sauté.")
else:
    print("Téléchargement de TOM2024 (raw)...")
    ds = load_dataset("Project-AgML/TOM2024_disease_classification", "raw", split="train")
    ds.save_to_disk(str(tom_dir))

if target_ccmt.exists() and any(target_ccmt.rglob("*.jpg")):
    print("CCMT déjà présent dans data_local/raw/ccmt_raw, téléchargement sauté.")
else:
    print("Téléchargement de CCMT Raw Data...")
    ccmt_path = Path(kagglehub.dataset_download("irakozekelly/crop-pest-and-disease-dataset"))
    target_ccmt.parent.mkdir(parents=True, exist_ok=True)
    for sub in ccmt_path.rglob("Raw Data"):
        if sub.is_dir():
            shutil.copytree(sub, target_ccmt, dirs_exist_ok=True)
            break
print("Jeux bruts prêts !")


In [ ]:
# 7. Construction de terramind_v1 (35 448 images) et découpage stratifié train/val/test (80/10/10)
from pathlib import Path
import subprocess

if not Path("data_local/processed/terramind_v1/manifest.csv").is_file():
    print("Construction de terramind_v1 (environ 2 min)...")
    subprocess.run(["python3", "ml/training/build_merged_dataset.py"], check=True)
else:
    print("terramind_v1 déjà construite, étape sautée.")

print("Découpage stratifié train / val / test...")
subprocess.run(["python3", "ml/training/split_dataset.py"], check=True)


In [ ]:
# 8. Fine-tuning du modèle Hugging Face pré-entraîné PlantVillage sur terramind_v1 (FP16 sur GPU T4)
!python3 -u ml/training/train_classifier.py \
    --backbone linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification \
    --epochs-head 2 \
    --epochs-finetune 6 \
    --batch-size 64 \
    --lr 1e-4 \
    --leaf-only


In [ ]:
# 9. Affichage du rapport d'évaluation et téléchargement automatique vers ton PC
import json
from pathlib import Path

metrics = json.loads(Path("ml/eval/metrics.json").read_text(encoding="utf-8"))
print("=== RÉSULTATS SUR LE JEU DE TEST (terramind_v1) ===")
print("Exactitude globale :", metrics["test_metrics"]["exactitude_globale"])
print("Seuil incertain calibré :", metrics["seuil_incertain"])
print("Exactitude hors incertain :", metrics["test_metrics"]["exactitude_hors_incertain"])
print("Taux incertain :", metrics["test_metrics"]["taux_incertain"])
print("Par source :", json.dumps(metrics["test_metrics"]["par_source"], indent=2, ensure_ascii=False))
print("Par culture :", json.dumps(metrics["test_metrics"]["par_culture"], indent=2, ensure_ascii=False))

try:
    from google.colab import files
    files.download("ml/eval/metrics.json")
    files.download("modeles/diagnostic/classes.json")
    files.download("modeles/diagnostic/terramind_v1_model.pt")
except ImportError:
    print("Fichiers disponibles dans modeles/diagnostic/ et ml/eval/.")
